# LoRA from scratch

Full fine-tuning updates every weight in the model. LoRA freezes the base weight `W` and learns a low-rank update `BA` instead, so you train a tiny fraction of the parameters and still steer the model. This notebook builds a `LoRALinear` by hand, checks the parameter saving, then does the same thing with `peft` on a tiny GPT-2.

In [ ]:
import torch
from llm_finetune import config, model
from torch import nn

from common.dl import set_seed

set_seed(42)

## 1. The maths

A linear layer is `y = W x` with `W` of shape `(out, in)`. LoRA adds a low-rank update:

`dW = (alpha / r) * B @ A`

where `A` is `(r, in)` and `B` is `(out, r)`, with `r` much smaller than `in` or `out`. The adapted layer computes `y = W x + (alpha / r) * B (A x)`. At init `B = 0`, so `dW = 0` and the model starts unchanged. `A` is small random, `B` is zeros. Only `A` and `B` train; `W` stays frozen.

## 2. From scratch: `LoRALinear`

Wrap a frozen `nn.Linear` and add the trainable factors `A` and `B`. The base weight has `requires_grad = False`, so gradients only flow into the adapter.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: int) -> None:
        super().__init__()
        self.base = base
        self.base.weight.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)
        self.scaling = alpha / r
        self.a = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.b = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        update = (x @ self.a.T) @ self.b.T
        return self.base(x) + self.scaling * update


set_seed(42)
base = nn.Linear(64, 64)
lora = LoRALinear(base, r=8, alpha=16)

At init `B = 0`, so the LoRA layer must return exactly the base output. Once we write non-zero values into `B`, the output shifts.

In [ ]:
x = torch.randn(4, 64)

same = torch.allclose(lora(x), base(x))
print(f'equals base at init: {same}')

with torch.no_grad():
    lora.b.add_(0.1)
shift = (lora(x) - base(x)).abs().mean().item()
print(f'mean shift after setting B: {shift:.4f}')

## 3. Parameter count

The frozen base holds `in * out` weights. The adapter holds `r * (in + out)`. For a 64 by 64 layer at `r = 8` that is a small slice of the whole.

In [ ]:
frozen = sum(p.numel() for p in lora.base.parameters())
trainable = sum(p.numel() for p in [lora.a, lora.b])
pct = 100 * trainable / (frozen + trainable)

print(f'frozen base params: {frozen}')
print(f'trainable LoRA params: {trainable}')
print(f'trainable share: {pct:.2f}%')

## 4. Library version: `peft` on the tiny GPT-2

`model.add_lora` wraps the base model with `peft.get_peft_model` using a `LoraConfig` built from `TrainConfig`. `resolve_target_modules` picks which layers get an adapter. For GPT-2 that is `c_attn`, the fused query-key-value projection.

In [ ]:
set_seed(42)
base_model, tokenizer = model.tiny_causal_lm(seed=42)
cfg = config.TrainConfig(lora_rank=8, lora_alpha=16, lora_dropout=0.0)

targets = model.resolve_target_modules(base_model, cfg)
print(f'target modules: {targets}')

lora_model = model.add_lora(base_model, cfg)

Count trainable versus total parameters the same way `peft` does: a parameter trains when `requires_grad` is true. The ratio matches the from-scratch idea, only a small percentage of the model moves.

In [ ]:
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in lora_model.parameters())
pct = 100 * trainable / total

print(f'trainable params: {trainable}')
print(f'total params: {total}')
print(f'trainable share: {pct:.2f}%')

## 5. What breaks and the two knobs

Rank `r` sets capacity. A higher `r` can fit a harder adaptation but adds `r * (in + out)` parameters per layer, so the saving shrinks. `alpha` scales the update through `alpha / r`; raising it makes the adapter act sooner during training. A common default keeps `alpha` at twice `r`.

`target_modules` depends on the architecture. GPT-2 fuses attention into `c_attn`, so that is the target here. Qwen and Llama split attention, so there you target `q_proj` and `v_proj`. Point LoRA at a name the model does not have and `peft` adapts nothing.